# MAX_SECOM_DS01 — XGBoost Classification 평가 예제

Croissant metadata(`MAX_SECOM_DS01.jsonld`)에 정의된 **test RecordSet**을 로더로 읽고, 저장된 XGBoost 모델(`secom_xgb.json`)로 Pass/Fail을 평가하는 예제입니다.

### 데이터/모델 기준
- Croissant Dataset: `MAX_SECOM_DS01`
- 평가 split: `test`
- 라벨: `Pass=-1`, `Fail=1` (metadata 기준)
- 학습 코드에서는 `Fail(1)=양성`으로 변환하여 `Pass=0`, `Fail=1`로 XGBoost를 학습
- `Time`은 Croissant에서 `Text`이므로 numeric-only loader에서 자동 제외
- 모델은 590개 입력 feature를 기대함

> **중요:** 결측치 처리, scaling, feature selection/engineering 등 모델 특화 전처리는 공통 evaluator가 수행하지 않습니다. 이번 모델의 학습 코드에는 별도 전처리가 없었으므로 여기서는 label mapping만 동일하게 재현합니다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import xgboost as xgb

from loader.data_loader import build_tabular_dataset

In [3]:
# 파일 경로 설정
JSONLD = 'MAX_SECOM_DS01.jsonld'
MODEL_PATH = 'secom_xgb.json'
SPLIT = 'test'

print('JSONLD :', Path(JSONLD).resolve())
print('MODEL  :', Path(MODEL_PATH).resolve())

JSONLD : /workspace/github/samples/anomaly_detection/MAX_SECOM_DS01.jsonld
MODEL  : /workspace/github/samples/anomaly_detection/secom_xgb.json


## 1. 저장된 XGBoost 모델 로드

학습 당시 사용한 `XGBClassifier` 설정을 다시 학습하지 않고, 저장된 모델을 그대로 불러옵니다.

In [4]:
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

print('Model loaded successfully.')
print('Expected input features:', model.n_features_in_)
print('Model classes:', model.classes_)

Model loaded successfully.
Expected input features: 590
Model classes: [0 1]


## 2. Croissant metadata에서 test 데이터 로드

`build_tabular_dataset()`이 metadata의 `records-test` RecordSet과 numeric field를 기준으로 `X, y, feature_names`를 구성합니다.

metadata상 `records-test/label`은 원본 CSV의 `Pass/Fail` 컬럼에서 추출되며, `Time`은 `Text` 타입입니다. 따라서 `numeric_only=True`를 사용하면 모델 입력에서는 `Time`이 제외됩니다.

In [6]:
X_raw, y_raw, feature_names = build_tabular_dataset(
    JSONLD,
    split=SPLIT,
    numeric_only=True,
)

print('X shape:', X_raw.shape)
print('y shape:', y_raw.shape)
print('First 10 features:', feature_names[:10])
print('Last 5 features:', feature_names[-5:])
print('Raw labels:', np.unique(y_raw, return_counts=True))

/usr/local/lib/python3.11/dist-packages/mlcroissant/_src/operation_graph/operations/read.py:274: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  file_content[FileProperty.filepath] = file.filepath  # type: ignore[call-overload]
/usr/local/lib/python3.11/dist-packages/mlcroissant/_src/operation_graph/operations/read.py:275: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  file_content[FileProperty.filename] = file.filename  # type: ignore[call-overload]
/usr/local/lib/python3.11/dist-packages/mlcroissant/_src/operation_graph/operation

X shape: (312, 590)
y shape: (312,)
First 10 features: ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', 'f_5', 'f_6', 'f_7', 'f_8', 'f_9']
Last 5 features: ['f_585', 'f_586', 'f_587', 'f_588', 'f_589']
Raw labels: (array([-1,  1]), array([268,  44]))


## 3. 모델 학습과 동일한 label mapping 적용

Croissant metadata의 클래스 정의는 `Pass=-1`, `Fail=1`입니다. 학습 코드에서는 다음과 같이 변환했습니다.

```python
y = (y.astype(int) == 1).astype(int)
```

따라서 평가에서도 동일하게 `Pass=0`, `Fail=1`로 변환합니다. 이 변환은 **데이터셋 공통 로더가 아니라 해당 모델의 학습 방식에 종속된 처리**이므로 Notebook에서 수행합니다.

In [ ]:
y_test = (y_raw.astype(int) == 1).astype(int)

print('Mapped labels:')
print('  Pass(-1) -> 0')
print('  Fail( 1) -> 1')
print('Class distribution:', np.unique(y_test, return_counts=True))

Mapped labels:
  Pass(-1) -> 0
  Fail( 1) -> 1
Class distribution: (array([0, 1]), array([268,  44]))


## 4. 모델 입력 feature 정합성 확인

저장된 모델은 590개 feature를 기대합니다. metadata에서 `Time`을 제외한 numeric feature가 동일한 개수인지 먼저 검증합니다.

이번 XGBoost 모델은 저장 시 feature name을 보존하지 않았기 때문에, **feature 개수만 확인해서는 순서까지 보장할 수 없습니다.** 따라서 metadata의 동일한 field 순서를 사용해 train/test를 구성해야 합니다. 이 Notebook에서는 학습 코드와 동일한 `build_tabular_dataset()`을 사용합니다.

In [ ]:
if X_raw.shape[1] != model.n_features_in_:
    raise ValueError(
        f'Feature 개수 불일치: dataset={X_raw.shape[1]}, '
        f'model={model.n_features_in_}'
    )

print(f'Feature check OK: {X_raw.shape[1]} features')

Feature check OK: 590 features


## 5. 공통 Classification Evaluator로 평가

Evaluator에는 **모델 입력이 완료된 `X`와 정답 `y`**를 전달합니다. 모델별 전처리가 필요한 경우 이 지점 앞에서 Notebook에서 수행하면 됩니다.

이번 모델은 학습 코드에 별도의 결측치 대체/스케일링/feature selection이 없었으므로 `X_model = X_raw`로 사용합니다.

In [9]:
def evaluate_classification_model(model, X, y_true, labels=None):
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(model.predict(X)).ravel()

    if len(y_true) != len(y_pred):
        raise ValueError("y_true와 y_pred의 길이가 다릅니다.")

    labels = np.unique(y_true) if labels is None else np.asarray(labels)
    average = "binary" if len(labels) == 2 else "weighted"

    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(
            y_true, y_pred, average=average, zero_division=0
        )),
        "recall": float(recall_score(
            y_true, y_pred, average=average, zero_division=0
        )),
        "f1": float(f1_score(
            y_true, y_pred, average=average, zero_division=0
        )),
        "confusion_matrix": confusion_matrix(
            y_true, y_pred, labels=labels
        ),
        "classification_report": classification_report(
            y_true, y_pred,
            labels=labels,
            zero_division=0,
            output_dict=True,
        ),
        "labels": labels.tolist(),
        "roc_auc": None,
    }

    if len(labels) == 2 and hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X)[:, 1]
            result["roc_auc"] = float(
                roc_auc_score(y_true == labels[-1], y_prob)
            )
        except ValueError:
            pass

    return result

In [11]:
X_model = X_raw

result = evaluate_classification_model(
    model,
    X_model,
    y_test,
    labels=[0, 1],
)

print(f"=== Classification 평가 결과 (test 셋) ===")
print(f"  Accuracy  : {result['accuracy']:.4f}")
print(f"  Precision : {result['precision']:.4f}")
print(f"  Recall    : {result['recall']:.4f}")
print(f"  F1        : {result['f1']:.4f}")
if result.get("roc_auc") is not None:
    print(f"  ROC-AUC   : {result['roc_auc']:.4f}")
print("  Confusion Matrix:")
print(result["confusion_matrix"])

=== Classification 평가 결과 (test 셋) ===
  Accuracy  : 0.8077
  Precision : 0.2143
  Recall    : 0.1364
  F1        : 0.1667
  ROC-AUC   : 0.6288
  Confusion Matrix:
[[246  22]
 [ 38   6]]
